# SemMap / ConceptNet-100k: nSBM → четыре теста самоподобия

**Два независимых этапа с восстановлением из Google Drive:**

**A. nSBM-only (CPU/OpenMP):** подготовленный ConceptNet-100k TSV → полный направленный типизированный граф → несколько запусков nested DC-SBM → `fit_data/` + `nsbm_partitions.npz`. Никаких VF2, Wishart и Huffman — они не нужны для измерения самоподобия.

**B. Самоподобие (CPU/SciPy):** сравнение структурных направленных мотивов, нормированных графонных профилей отношений, линейных медленных мод Хаккена и независимого box-covering/D(q), с двумя контрольными ансамблями и частичным RESUME.

Если **правильные 100k nSBM разбиения уже сохранены**, этап A пропускается, и для B достаточно обычного Colab CPU. Для этапа A желательно **High-RAM**. GPU T4/L4 *не ускоряет* graph-tool nSBM: он работает на CPU.

**Научное ограничение:** число блоков на 100k может отличаться от 22 → 7 → 3; графонный AUC при использовании full-graph nSBM — условная диагностика с утечкой данных, а не независимый прогноз. Пять радиусов покрытий не доказывают фрактальность.


In [ ]:
# 1. Настройки. Для нового исследования — NEW, после обрыва — RESUME.
from pathlib import Path
REPO_URL = "https://github.com/SemanticMap/semgraphex.git"
REPO_BRANCH = "feature/nsbm-multiscale-self-similarity"
SOURCE_REVISION = "6a19a56c240bb7a57b4b3bfe722ff064c2e4a3c8"
RUN_MODE = "NEW"  # NEW / RESUME; после прерывания измените только это
SEED = 42
CPU_BUDGET = 12
NSBM_RESTARTS = 3  # независимые nested DC-SBM; увеличить при подтверждающих опытах
NULL_REPEATS = 19  # min empirical p=0.05; 99+ для подтверждающей серии
MAX_SIMULTANEOUS_NULLS = 2  # ограничивает RAM, не число ядер для nSBM
MOTIF_SAMPLES = 1500
PAIR_SAMPLES = 10000
BOX_REPEATS = 2
BOX_RADII = (0,1,2,3,4)  # для 100k начните с пяти масштабов
SLOW_MODES = 3
SYNC_EVERY_SECONDS = 120
DRIVE_PROJECT = Path("/content/drive/MyDrive/SemanticMap")
FIT_RUN_ID = "nsbm-fit-cn100k-01"
RUN_ID = "nsbm-selfsimilarity-cn100k-01"
FIT_STORE = DRIVE_PROJECT/"colab/hypothesis/runs"/FIT_RUN_ID
RESULT_STORE = DRIVE_PROJECT/"colab/hypothesis/runs"/RUN_ID
# Исходный ФАЙЛ — пять полей ConceptNet TSV (или .gz/.bz2/.xz);
# он ДОЛЖЕН порождать ровно 100000 разных понятий.
# Исходный файл проверен: 100000 понятий, 536128 assertions, 6 отношений.
PREPARED_100K_TSV = DRIVE_PROJECT/"colab/wishart/data/conceptnet_en_100k.tsv"
# Необязательно: оригинальные или заранее отфильтрованные assertions ConceptNet.
# Если PREPARED_100K_TSV отсутствует и RAW_SOURCE_TSV задан, ноутбук
# сформирует 100k по relation-stratified BFS. Это НЕ равномерная выборка.
RAW_SOURCE_TSV = None  # when None, automatically download official ConceptNet 5.7
CONCEPTNET_URL = "https://s3.amazonaws.com/conceptnet/downloads/2019/edges/conceptnet-assertions-5.7.0.csv.gz"
RESERVOIR_ASSERTIONS = 600000  # sampled over the FULL English stream, no file-prefix bias
SAMPLE_RELATIONS = "RelatedTo,IsA,PartOf,UsedFor,HasProperty,HasA"
MAX_ELIGIBLE_ASSERTIONS = 2000000  # safety cap; превышение = отказ, не prefix
assert RUN_MODE in ("NEW","RESUME")


In [ ]:
# 2. Ресурсы Colab, планируем число CPU/память и локальный SSD
import os, sys, subprocess, shutil, time, json, hashlib
def allowed_cpus():
    n=os.cpu_count() or 1
    if hasattr(os,"sched_getaffinity"):
        n=min(n,len(os.sched_getaffinity(0)))
    try:
        q,p=Path("/sys/fs/cgroup/cpu.max").read_text().split()
        if q!="max": n=min(n,max(1,(int(q)+int(p)-1)//int(p)))
    except (ValueError,OSError): pass
    return max(1,n)
CPU_EFFECTIVE=min(CPU_BUDGET,allowed_cpus())
DISK_GIB=shutil.disk_usage("/content").free/2**30
print({"python":sys.version.split()[0],"cpu_available":allowed_cpus(),
       "cpu_budget":CPU_EFFECTIVE,"disk_free_GiB":round(DISK_GIB,1)})
if DISK_GIB < 9: raise RuntimeError("Для 100k и graph-tool требуется ~9 GiB локального диска")


In [ ]:
# 3. Хранилище и локальные рабочие папки
from google.colab import drive
drive.mount("/content/drive")
SCRATCH=Path("/content/semmap_selfsim_100k")
REPO=Path("/content/semmap_selfsim_100k_repo")
BIN_ROOT=SCRATCH/"micromamba_bin"
ENV_ROOT=SCRATCH/"graph_tool_env"
FIT_LOCAL=SCRATCH/"runs"/FIT_RUN_ID
RESULT_LOCAL=SCRATCH/"runs"/RUN_ID
# Keep .gz/.bz2/.xz suffix when a prefiltered prepared input is compressed.
STAGED_TSV=SCRATCH/"input"/(
    PREPARED_100K_TSV.name if PREPARED_100K_TSV.is_file()
    else "conceptnet_en_100k_relations.tsv"
)
SCRATCH.mkdir(parents=True,exist_ok=True)

def sync_directory(src, dest, final=False):
    dest.mkdir(parents=True,exist_ok=True)
    for original in sorted(src.rglob("*")):
        if (not original.is_file() or original.name=="COMPLETED"
            or original.name.endswith((".tmp",".partial"))):
            continue
        copy_to=dest/original.relative_to(src)
        copy_to.parent.mkdir(parents=True,exist_ok=True)
        if (not copy_to.exists() or
            copy_to.stat().st_size!=original.stat().st_size or
            copy_to.stat().st_mtime_ns!=original.stat().st_mtime_ns):
            shutil.copy2(original,copy_to)
    if final and (src/"COMPLETED").is_file():
        shutil.copy2(src/"COMPLETED",dest/"COMPLETED")

def run_logged(command, *, cwd, env, local, store, log_name):
    log=SCRATCH/log_name
    with log.open("w",encoding="utf-8") as stream:
        proc=subprocess.Popen(command,cwd=cwd,env=env,stdout=stream,
                              stderr=subprocess.STDOUT)
        try:
            while True:
                try:
                    proc.wait(timeout=SYNC_EVERY_SECONDS);break
                except subprocess.TimeoutExpired:
                    sync_directory(local,store)
                    print("Checkpoint synced:",store.name)
        except KeyboardInterrupt:
            proc.terminate();proc.wait();raise
        finally:
            sync_directory(local,store,final=proc.poll()==0)
    if proc.returncode:
        print("\n".join(log.read_text(errors="replace").splitlines()[-70:]))
        raise RuntimeError(f"Subprocess {log_name} failed: {proc.returncode}")
    assert (store/"COMPLETED").is_file(), "COMPLETED marker missing"
    print("\n".join(log.read_text(errors="replace").splitlines()[-12:]))


In [ ]:
# 4. Один закреплённый Git SHA, host Python для 4 методов; без CUDA-установки
if not (REPO/".git").is_dir():
    subprocess.run(["git","clone","--depth","100","--branch",
                    REPO_BRANCH,REPO_URL,str(REPO)],check=True)
else:
    subprocess.run(["git","-C",str(REPO),"fetch","--depth","100",
                    "origin",REPO_BRANCH],check=True)
subprocess.run(["git","-C",str(REPO),"checkout","--detach",SOURCE_REVISION],check=True)
assert subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],
                                text=True).strip()==SOURCE_REVISION
subprocess.run([sys.executable,"-m","pip","install","-q","--no-deps",
                "-e",str(REPO)],check=True)
# Colab обычно включает scipy/sklearn/matplotlib. Не обновляем весь базовый стек.
import scipy, sklearn, numpy, matplotlib
try: import pytest
except ImportError:
    subprocess.run([sys.executable,"-m","pip","install","-q","pytest"],check=True)
check_env=dict(os.environ,OPENBLAS_NUM_THREADS="1",MKL_NUM_THREADS="1",
               NUMEXPR_NUM_THREADS="1",PYTHONHASHSEED="0",
               PYTHONWARNINGS="ignore:The hashes produced for directed graphs changed in version v3.5:UserWarning")
print("Pinned revision:",SOURCE_REVISION)


In [ ]:
# 5. Reuse: не требуем входной TSV, если готовый nSBM-fit уже есть в Drive
# COMPLETED can survive on Drive while fit_data is missing in ephemeral /content.
# Stage each file atomically, verify original node IDs, graph digest and nSBM
# levels, and only then decide whether expensive graph-tool fitting can be skipped.
from semmap_haken.hypothesis.selfsim_colab_io import stage_fit_snapshot
FIT_STATUS=stage_fit_snapshot(FIT_STORE,FIT_LOCAL,expected_vertices=100000)
FIT_ALREADY_DONE=FIT_STATUS["ready"]
print("nSBM fit ready:",FIT_ALREADY_DONE,
      "verified block counts:",FIT_STATUS.get("blocks"),
      "staged files:",FIT_STATUS["staged_files"],
      "reason:",FIT_STATUS["reason"])
if not FIT_ALREADY_DONE and FIT_STATUS["missing"]:
    print("Missing fit files:",FIT_STATUS["missing"])
if not FIT_ALREADY_DONE:
    STAGED_TSV.parent.mkdir(parents=True,exist_ok=True)
    if PREPARED_100K_TSV.is_file():
        shutil.copy2(PREPARED_100K_TSV,STAGED_TSV)
    else:
        if RAW_SOURCE_TSV is None:
            raw_path=SCRATCH/"input"/"conceptnet-assertions-5.7.0.csv.gz"
            if not raw_path.is_file():
                print("Downloading official ConceptNet 5.7 assertions (large full gzip)...")
                pending=raw_path.with_suffix(raw_path.suffix+".part")
                subprocess.run(["curl","-fL","--retry","4","--continue-at","-",
                                "-o",str(pending),CONCEPTNET_URL],check=True)
                os.replace(pending,raw_path)
        else:
            raw_path=Path(RAW_SOURCE_TSV)
            if not raw_path.is_file(): raise FileNotFoundError(raw_path)
        # The official full dump exceeds the bounded BFS input cap.
        # Reservoir-sample eligible English assertions across the COMPLETE file.
        # A documented two-stage sample is scientifically more honest than
        # taking the first MAX_ELIGIBLE_ASSERTIONS records.
        staged_source=SCRATCH/"input"/"eligible_en_bounded.tsv"
        if not staged_source.is_file():
            cmd=[sys.executable,"-m",
                 "semmap_haken.hypothesis.conceptnet_bounded_source",
                 "--input",str(raw_path),"--output",str(staged_source),
                 "--maximum",str(RESERVOIR_ASSERTIONS),"--seed",str(SEED),
                 "--relations",SAMPLE_RELATIONS]
            subprocess.run(cmd,cwd=REPO,env=check_env,check=True)
        cmd=[sys.executable,"-m","semmap_haken.hypothesis.conceptnet_10k",
             "--input",str(staged_source),"--output",str(STAGED_TSV),
             "--nodes","100000","--seed",str(SEED),
             "--relations",SAMPLE_RELATIONS,
             "--max-assertions",str(MAX_ELIGIBLE_ASSERTIONS)]
        subprocess.run(cmd,cwd=REPO,env=check_env,check=True)
        PREPARED_100K_TSV.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(STAGED_TSV,PREPARED_100K_TSV)
        shutil.copy2(STAGED_TSV.with_name(STAGED_TSV.name+".selection.json"),
                     PREPARED_100K_TSV.with_name(PREPARED_100K_TSV.name+".selection.json"))
    def sha_file(path):
        h=hashlib.sha256()
        with Path(path).open("rb") as src:
            for block in iter(lambda:src.read(1<<20),b""):h.update(block)
        return h.hexdigest()
    if sha_file(STAGED_TSV)!=sha_file(PREPARED_100K_TSV):
        raise RuntimeError("Drive TSV differs from local copy")
    print("100k input staged locally:",STAGED_TSV,
          "MiB:",round(STAGED_TSV.stat().st_size/2**20,1))


In [ ]:
# 6. Если нет готовых разбиений, fit ONLY nested-DC-SBM (graph-tool CPU/OpenMP).
# Для расчёта nSBM не нужен GPU; high-RAM полезнее T4/L4.
if not FIT_ALREADY_DONE:
    BIN_ROOT.mkdir(parents=True,exist_ok=True)
    MAMBA=BIN_ROOT/"bin"/"micromamba"
    FIT_PY=ENV_ROOT/"bin"/"python"
    if not MAMBA.is_file():
        subprocess.run(["bash","-lc","set -euo pipefail; curl -fsSL "
          "https://micro.mamba.pm/api/micromamba/linux-64/latest | "
          f"tar -xj -C {BIN_ROOT} bin/micromamba"],check=True)
    mm_env=dict(os.environ,MAMBA_ROOT_PREFIX=str(SCRATCH/"micromamba_root"))
    if not FIT_PY.is_file():
        subprocess.run([str(MAMBA),"create","-y","-p",str(ENV_ROOT),
            "-c","conda-forge","--strict-channel-priority",
            "python=3.12","graph-tool-base=2.98","numpy","scipy","scikit-learn",
            "networkx","pyyaml","psutil","threadpoolctl","pip"],check=True,env=mm_env)
    subprocess.run([str(FIT_PY),"-m","pip","install","--no-deps",
                    "-e",str(REPO)],check=True)
    subprocess.run([str(FIT_PY),"-c",
                    "import graph_tool.all as gt; print('graph-tool:',gt.__version__)"],
                    check=True)
    stage_mode="RESUME" if (FIT_LOCAL/"manifest.json").is_file() else "NEW"
    fit_command=[str(FIT_PY),"-m","semmap_haken.hypothesis.nsbm_fit_only",
        "--input",str(STAGED_TSV),"--output",str(FIT_LOCAL),
        "--vertices","100000","--mode",stage_mode,
        "--seed",str(SEED),"--restarts",str(NSBM_RESTARTS),
        "--cpu-budget",str(CPU_EFFECTIVE)]
    run_logged(fit_command,cwd=REPO,env=check_env,local=FIT_LOCAL,
               store=FIT_STORE,log_name=FIT_RUN_ID+".log")
else:
    print("Reuse complete nSBM fit; skipping expensive graph-tool learning")


In [ ]:
# 7. Восстановить и прочитать подтвержденный nSBM fit за одну операцию
# Пересчитываем пути явно: предыдущие ячейки могли быть запущены в другом порядке.
from pathlib import Path
import json, hashlib

FIT_STORE = Path("/content/drive/MyDrive/SemanticMap/colab/hypothesis/runs") / FIT_RUN_ID
FIT_LOCAL = Path("/content/semmap_selfsim_100k/runs") / FIT_RUN_ID

from semmap_haken.hypothesis.selfsim_colab_io import load_verified_fit_bundle
g, levels, metadata, fit_summary, FIT_STATUS = load_verified_fit_bundle(
    FIT_STORE, FIT_LOCAL, expected_vertices=100000
)
GRAPH_DIR = FIT_LOCAL / "fit_data"
PARTITIONS = FIT_LOCAL / "nsbm_partitions.npz"
SOURCE_MANIFEST = FIT_LOCAL / "manifest.json"
print("Verified file:", GRAPH_DIR / "ids.json",
      "bytes:", (GRAPH_DIR/"ids.json").stat().st_size)
print("Verified nSBM blocks:", FIT_STATUS["blocks"])

def sha_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as src:
        for block in iter(lambda: src.read(1<<20), b""):
            h.update(block)
    return h.hexdigest()
assert sha_file(PARTITIONS) == fit_summary["partitions_sha256"]
if PREPARED_100K_TSV.is_file():
    assert sha_file(PREPARED_100K_TSV) == metadata["source_tsv_sha256"], (
        "The saved nSBM fit belongs to a different ConceptNet TSV."
    )
print({"vertices":g.vertex_count,"assertions":g.edge_count,
       "relations":g.relation_names,"nested_blocks":[len(set(p)) for p in levels]})


In [ ]:
# 8. Тесты перед дорогими вычислениями
subprocess.run([sys.executable,"-m","pytest","-q",
                "tests/test_selfsim_four.py","tests/test_nsbm_fit_only.py",
                "tests/test_selfsim_colab_io.py"],
               cwd=REPO,env=check_env,check=True)


In [ ]:
# 9. Четыре теста + контрольные графы. GPU не требуется.
if RESULT_STORE.is_dir():
    shutil.copytree(RESULT_STORE,RESULT_LOCAL,dirs_exist_ok=True)
if RUN_MODE=="NEW" and RESULT_LOCAL.exists() and any(RESULT_LOCAL.iterdir()):
    raise ValueError("NEW refuses to overwrite self-similarity results; "
                     "set RUN_MODE='RESUME' or a new RUN_ID")
if RUN_MODE=="RESUME" and not (RESULT_LOCAL/"manifest.json").is_file():
    raise FileNotFoundError("No self-similarity checkpoint to RESUME")
command=[sys.executable,"-m","semmap_haken.hypothesis.selfsim_experiment",
         "--graph-dir",str(GRAPH_DIR),"--partitions",str(PARTITIONS),
         "--source-manifest",str(SOURCE_MANIFEST),
         "--output",str(RESULT_LOCAL),"--mode",RUN_MODE,
         "--seed",str(SEED),"--null-repeats",str(NULL_REPEATS),
         "--jobs",str(min(MAX_SIMULTANEOUS_NULLS,CPU_EFFECTIVE)),
         "--motif-samples",str(MOTIF_SAMPLES),"--pair-samples",str(PAIR_SAMPLES),
         "--box-repeats",str(BOX_REPEATS),
         "--box-radii",",".join(map(str,BOX_RADII)),"--slow-modes",str(SLOW_MODES)]
run_logged(command,cwd=REPO,env=check_env,local=RESULT_LOCAL,
           store=RESULT_STORE,log_name=RUN_ID+".log")


In [ ]:
# 10. Сравнение четырёх независимых диагностик и нулевых моделей
import pandas as pd
report=json.loads((RESULT_STORE/"report.json").read_text(encoding="utf-8"))
print("ConceptNet:",report["counts"],"nSBM blocks:",report["block_counts"])
rows=[]
for metric,observed in report["scores"].items():
    row={"diagnostic":metric,"observed":observed}
    for control,values in report["controls"].items():
        stats=values.get(metric,{})
        row[control+"_mean"]=stats.get("null_mean")
        row[control+"_p_upper"]=stats.get("p_upper")
    rows.append(row)
display(pd.DataFrame(rows))
print("WARNING:",report["inference_warning"])
print("WARNING:",report["scaling_warning"])
print("Report:",RESULT_STORE/"report.json")


In [ ]:
# 11. Отдельные графики; сравнение 10k и 100k делается по метрикам, не по ID блоков
import matplotlib.pyplot as plt
observed=report["observed"]
s=observed["structural"]["adjacent_motif_js"]
if s:
    fig,ax=plt.subplots(figsize=(7,3.4))
    ax.plot([x["from"]+" → "+x["to"] for x in s],
            [x["js_divergence"] for x in s],marker="o")
    ax.set(ylabel="JS bits",title="1. Directed motif similarity")
    ax.tick_params(axis="x",rotation=25)
    plt.tight_layout();plt.show()
k=observed["graphon"]["adjacent_kernel"]
if k:
    fig,ax=plt.subplots(figsize=(7,3.4))
    ax.plot([str(x["fine_blocks"])+"→"+str(x["coarse_blocks"]) for x in k],
            [x["log_intensity_js"] for x in k],marker="o")
    ax.set(ylabel="JS bits",title="2. Typed step-graphon profile")
    plt.tight_layout();plt.show()
dy=observed["haken"]["levels"]
fig,ax=plt.subplots(figsize=(7,3.4))
ax.plot([str(x["blocks"]) for x in dy],
        [x["slow_subspace_overlap"] for x in dy],marker="o")
ax.set(xlabel="number of blocks",ylim=(0,1.05),
       ylabel="subspace overlap",title="3. Haken linearized slow-mode stability")
plt.tight_layout();plt.show()
box=observed["boxes"]["rows"]
rs=sorted(set(int(x["radius"]) for x in box))
x=[2*r+1 for r in rs]
y=[sum(row["box_count"] for row in box if row["radius"]==r)/
   sum(row["radius"]==r for row in box) for r in rs]
fig,ax=plt.subplots(figsize=(7,3.4))
ax.loglog(x,y,marker="o")
ax.set(xlabel="2r+1",ylabel="greedy box count",
       title="4. Independent box-covering upper bound")
plt.tight_layout();plt.show()


## Как интерпретировать

- Размеры блоков и порядок nSBM могут **не совпасть** с 10k. Сравнивайте нормированные распределения и параметры, а не номера блоков.
- Структурная устойчивость: направленные мотивы и JS *относительно контролей*.
- Графон: веса масс и отдельные типы связей; `holdout_auc_conditional` нельзя трактовать как независимый прогноз из-за обучения nSBM на полном графе.
- Хаккен: этот тест применяет симметризованный линейный оператор, не утверждает нелинейный принцип подчинения.
- Box-covering: это эвристическая верхняя оценка минимального числа коробок; степенная зависимость по 5 радиусам сама по себе не доказывает самоподобие или мультифрактальность.
- `NULL_REPEATS=19` даёт разрешение p не лучше 0.05, и нет поправки на множественную проверку. Для доказательных выводов нужны независимые nSBM-подгонки и 99+ нулей (в новом RUN_ID).
- **После обрыва:** сохрани те же `SOURCE_REVISION`, `FIT_RUN_ID`, `RUN_ID`, `SEED`, настройки и поставь `RUN_MODE="RESUME"`. Уже опубликованная модель nSBM повторно не обучается.
